# Feature Extraction with Spark MLlib

In [2]:
!pip install findspark

Defaulting to user installation because normal site-packages is not writeable


Перезапустить kernel после установки!

In [3]:
import findspark
findspark.init()

In [4]:
!hdfs dfs -ls

Found 2 items
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 16:26 .sparkStaging
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 16:26 data


In [521]:
!hdfs dfs -ls /user/ubuntu/data

Found 44 items
-rw-r--r--   1 ubuntu hadoop 2807409271 2026-06-14 09:25 /user/ubuntu/data/2019-08-22.txt
-rw-r--r--   1 ubuntu hadoop 2854479008 2026-06-14 09:14 /user/ubuntu/data/2019-09-21.txt
-rw-r--r--   1 ubuntu hadoop 2895460543 2026-06-14 09:30 /user/ubuntu/data/2019-10-21.txt
-rw-r--r--   1 ubuntu hadoop 2939120942 2026-06-14 09:20 /user/ubuntu/data/2019-11-20.txt
-rw-r--r--   1 ubuntu hadoop 2995462277 2026-06-14 09:29 /user/ubuntu/data/2019-12-20.txt
-rw-r--r--   1 ubuntu hadoop 2994906767 2026-06-14 09:27 /user/ubuntu/data/2020-01-19.txt
-rw-r--r--   1 ubuntu hadoop 2995431240 2026-06-14 09:29 /user/ubuntu/data/2020-02-18.txt
-rw-r--r--   1 ubuntu hadoop 2995176166 2026-06-14 09:27 /user/ubuntu/data/2020-03-19.txt
-rw-r--r--   1 ubuntu hadoop 2996034632 2026-06-14 09:26 /user/ubuntu/data/2020-04-18.txt
-rw-r--r--   1 ubuntu hadoop 2995666965 2026-06-14 09:15 /user/ubuntu/data/2020-05-18.txt
-rw-r--r--   1 ubuntu hadoop 2994699401 2026-06-14 09:15 /user/ubuntu/data/2020-06-17

In [522]:
!hdfs dfs -ls data

Found 44 items
-rw-r--r--   1 ubuntu hadoop 2807409271 2026-06-14 09:25 data/2019-08-22.txt
-rw-r--r--   1 ubuntu hadoop 2854479008 2026-06-14 09:14 data/2019-09-21.txt
-rw-r--r--   1 ubuntu hadoop 2895460543 2026-06-14 09:30 data/2019-10-21.txt
-rw-r--r--   1 ubuntu hadoop 2939120942 2026-06-14 09:20 data/2019-11-20.txt
-rw-r--r--   1 ubuntu hadoop 2995462277 2026-06-14 09:29 data/2019-12-20.txt
-rw-r--r--   1 ubuntu hadoop 2994906767 2026-06-14 09:27 data/2020-01-19.txt
-rw-r--r--   1 ubuntu hadoop 2995431240 2026-06-14 09:29 data/2020-02-18.txt
-rw-r--r--   1 ubuntu hadoop 2995176166 2026-06-14 09:27 data/2020-03-19.txt
-rw-r--r--   1 ubuntu hadoop 2996034632 2026-06-14 09:26 data/2020-04-18.txt
-rw-r--r--   1 ubuntu hadoop 2995666965 2026-06-14 09:15 data/2020-05-18.txt
-rw-r--r--   1 ubuntu hadoop 2994699401 2026-06-14 09:15 data/2020-06-17.txt
-rw-r--r--   1 ubuntu hadoop 2995810010 2026-06-14 09:16 data/2020-07-17.txt
-rw-r--r--   1 ubuntu hadoop 2995995152 2026-06-14 09:31 data

In [7]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, DoubleType, FloatType, LongType, StringType, TimestampType
from datetime import timedelta


In [5]:
import warnings
warnings.filterwarnings('ignore')
spark_ui_port = 4040
app_name = "Otus"

from itertools import groupby

Создадим SparkSession

In [8]:
spark = (
    SparkSession
        .builder
        .appName("OTUS")
        .getOrCreate()
)
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)  # to pretty print pyspark.DataFrame in jupyter

In [526]:
spark

Загрузим данные для одного файла

In [527]:
file_data = '2020-05-18'

In [528]:
rdd = spark.sparkContext.textFile(f"data/{file_data}.txt")

header = rdd.filter(lambda x: x.startswith("#")).first()
columns = [c.strip() for c in header.replace("#", "").split("|")]

data = rdd.filter(lambda x: not x.startswith("#")) \
          .map(lambda x: x.split(","))

df_raw = spark.createDataFrame(data, schema=columns)

df_raw.show(15)

+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|tranaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_time_days|tx_fraud|tx_fraud_scenario|
+-------------+-------------------+-----------+-----------+---------+---------------+------------+--------+-----------------+
|    422936262|2020-05-18 07:54:59|          0|        612|    53.61|       23356499|         270|       0|                0|
|    422936263|2020-05-18 01:33:02|          0|        817|    53.81|       23333582|         270|       0|                0|
|    422936264|2020-05-18 06:36:11|          1|        632|    51.13|       23351771|         270|       0|                0|
|    422936265|2020-05-18 13:35:38|          3|        732|     3.32|       23376938|         270|       0|                0|
|    422936266|2020-05-18 05:34:00|          6|        920|    53.05|       23348040|         270|       0|           

# ФУНКЦИЯ ПО СТАТИСТИКЕ ДАТ

In [529]:
def analyze_dates(df, datetime_column='tx_datetime', date_format='yyyy-MM-dd HH:mm:ss'):
    """
    Функция для анализа дат в DataFrame

    Параметры:
    - df: Spark DataFrame
    - datetime_column: имя колонки с датой/временем
    - date_format: формат даты/времени

    Возвращает:
    - min_datetime: минимальная дата и время (строка в формате 'YYYY-MM-DD 00:00:00')
    - max_datetime: максимальная дата и время (строка в формате 'YYYY-MM-DD 00:00:00')
    - date_stats: DataFrame со статистикой по датам
    """

    print(f"\n{'='*60}")
    print(f"АНАЛИЗ ДАТ В КОЛОНКЕ: {datetime_column}")
    print(f"{'='*60}")

    # Преобразуем в timestamp и date
    df_temp = df.withColumn(datetime_column, F.to_timestamp(datetime_column, date_format))
    df_temp = df_temp.withColumn('tx_date', F.to_date(datetime_column))

    # Находим минимальную и максимальную дату
    min_date_row = df_temp.agg(F.min('tx_date')).collect()[0]
    max_date_row = df_temp.agg(F.max('tx_date')).collect()[0]

    min_date = min_date_row[0]
    max_date = max_date_row[0]

    # Преобразуем в строку с временем 00:00:00
    min_datetime = f"{min_date} 00:00:00" if min_date else None
    max_datetime = f"{max_date} 00:00:00" if max_date else None

    print(f"\nДИАПАЗОН ДАТ:")
    print(f"  Минимальная дата: {min_datetime}")
    print(f"  Максимальная дата: {max_datetime}")

    # Считаем количество дней между датами
    if min_date and max_date:
        days_diff = (max_date - min_date).days
        print(f"  Количество дней в диапазоне: {days_diff}")

    # Группируем по датам
    date_stats = df_temp.groupBy('tx_date').agg(
        F.count('*').alias('count'),
        F.count(F.when(F.col(datetime_column).isNull(), 1)).alias('null_count')
    ).orderBy('tx_date')

    # Статистика по датам
    print(f"\nСТАТИСТИКА ПО ДАТАМ:")
    print(f"  Всего уникальных дат: {date_stats.count():,}")
    print(f"  Всего записей: {df_temp.count():,}")

    # Показываем первые и последние даты
    print(f"\nПЕРВЫЕ 10 ДАТ (самые старые):")
    date_stats.show(10, truncate=False)

    print(f"\nПОСЛЕДНИЕ 10 ДАТ (самые новые):")
    date_stats.orderBy(F.desc('tx_date')).show(10, truncate=False)

    return min_datetime, max_datetime, date_stats

In [530]:
def analyze_column(df, column_name, target_type, timestamp_format='yyyy-MM-dd HH:mm:ss'):
    """
    Функция для проверки и анализа колонки без внесения изменений
    """

    type_mapping = {
        'int': IntegerType(),
        'double': DoubleType(),
        'float': FloatType(),
        'long': LongType(),
        'string': StringType(),
        'timestamp': TimestampType(),
    }

    if target_type not in type_mapping:
        raise ValueError(f"Неподдерживаемый тип: {target_type}. Используйте: {list(type_mapping.keys())}")

    spark_type = type_mapping[target_type]
    type_name = target_type.upper()

    print(f"\n{'='*60}")
    print(f"АНАЛИЗ КОЛОНКИ: {column_name}")
    print(f"{'='*60}")

    # Сначала преобразуем колонку в целевой тип
    if target_type == 'timestamp':
        df_converted = df.withColumn(
            f"{column_name}_converted",
            F.to_timestamp(F.col(column_name), timestamp_format)
        )
    else:
        df_converted = df.withColumn(
            f"{column_name}_converted",
            F.col(column_name).cast(spark_type)
        )

    # Теперь считаем статистику на преобразованной колонке
    total_count = df.count()
    null_count_original = df.filter(F.col(column_name).isNull()).count()
    empty_count = df.filter(F.col(column_name) == '').count()
    string_null_count = df.filter(F.col(column_name) == 'null').count()

    # NULL после преобразования (это и есть некорректные значения)
    null_count_after = df_converted.filter(F.col(f"{column_name}_converted").isNull()).count()

    # Валидные = те, что не NULL после преобразования
    valid_count = total_count - null_count_after

    print(f"\nСТАТИСТИКА:")
    print(f"   Всего записей: {total_count:,}")
    print(f"   Валидных для преобразования в {type_name}: {valid_count:,}")
    print(f"   Невалидных (стали NULL): {null_count_after - null_count_original:,}")
    print(f"   Из них исходные NULL: {null_count_original:,}")
    print(f"   Из них пустые строки: {empty_count:,}")
    print(f"   Из них строк 'null': {string_null_count:,}")

    # Показываем проблемные значения (которые стали NULL после преобразования)
    if null_count_after > null_count_original:
        problematic_df = df.filter(
            F.col(column_name).isNotNull() &
            df_converted.filter(F.col(f"{column_name}_converted").isNull()).select(column_name).isNotNull()
        )

        print(f"\nПРОБЛЕМНЫЕ ЗАПИСИ (первые 20):")
        problematic_df.select(column_name).show(20, truncate=False)

        print(f"\nУНИКАЛЬНЫЕ ПРОБЛЕМНЫЕ ЗНАЧЕНИЯ:")
        problematic_df.select(column_name).distinct().show(10, truncate=False)

    result_stats = {
        'total': total_count,
        'valid': valid_count,
        'invalid': null_count_after - null_count_original,
        'null_count': null_count_original,
        'empty_count': empty_count,
        'string_null_count': string_null_count,
        'problematic_count': null_count_after - null_count_original,
        'valid_percent': (valid_count / total_count * 100) if total_count > 0 else 0,
        'column_name': column_name,
        'target_type': target_type
    }

    # Очищаем временную колонку
    df_converted = df_converted.drop(f"{column_name}_converted")

    return result_stats, problematic_df if 'problematic_df' in locals() else None

In [531]:
def fix_24h_time(df, datetime_column='tx_datetime'):
    """
    Заменяет время 24:00:00 на 00:00:00 без изменения даты

    Параметры:
    - df: Spark DataFrame
    - datetime_column: имя колонки с датой/временем

    Возвращает:
    - DataFrame с исправленным временем
    """

    # Заменяем 24:00:00 на 00:00:00
    df_fixed = df.withColumn(
        datetime_column,
        F.regexp_replace(datetime_column, '24:00:00', '00:00:00')
    )

    # Статистика замен
    replaced_count = df.filter(F.col(datetime_column).contains('24:00:00')).count()

    if replaced_count > 0:
        print(f"Заменено {replaced_count} записей с 24:00:00 на 00:00:00")
        print(f"Дата осталась без изменений")
    else:
        print("Значений с 24:00:00 не найдено")

    return df_fixed

# ФУНКЦИЯ АНАЛИЗА КОЛОНКИ

In [532]:
def analyze_column(df, column_name, target_type, timestamp_format='yyyy-MM-dd HH:mm:ss'):
    """
    Функция для проверки и анализа колонки без внесения изменений

    Параметры:
    - df: Spark DataFrame
    - column_name: имя колонки для проверки
    - target_type: целевой тип ('int', 'double', 'float', 'long', 'string', 'timestamp')
    - timestamp_format: формат timestamp (по умолчанию 'yyyy-MM-dd HH:mm:ss')

    Возвращает:
    - stats: словарь со статистикой
    - problematic_df: DataFrame с проблемными записями
    """

    # Определяем тип для преобразования
    type_mapping = {
        'int': IntegerType(),
        'double': DoubleType(),
        'float': FloatType(),
        'long': LongType(),
        'string': StringType(),
        'timestamp': TimestampType(),
    }

    if target_type not in type_mapping:
        raise ValueError(f"Неподдерживаемый тип: {target_type}. Используйте: {list(type_mapping.keys())}")

    spark_type = type_mapping[target_type]
    type_name = target_type.upper()

    print(f"\n{'='*60}")
    print(f"АНАЛИЗ КОЛОНКИ: {column_name}")
    print(f"{'='*60}")

    # Создаем временную колонку для проверки валидности
    valid_col_name = f"{column_name}_valid"

    # Для timestamp используем специальную проверку через to_timestamp
    if target_type == 'timestamp':
        df_check = df.withColumn(valid_col_name,
            F.when(
                (F.col(column_name).isNull()) |
                (F.col(column_name) == '') |
                (F.col(column_name) == 'null') |
                (F.to_timestamp(F.col(column_name), timestamp_format).isNull()),
                False
            ).otherwise(True)
        )
    else:
        # Для остальных типов используем cast
        df_check = df.withColumn(valid_col_name,
            F.when(
                (F.col(column_name).isNull()) |
                (F.col(column_name) == '') |
                (F.col(column_name) == 'null') |
                (F.col(column_name).cast(spark_type).isNull()),
                False
            ).otherwise(True)
        )

    # Статистика
    stats = df_check.agg(
        F.count('*').alias('total'),
        F.sum(F.when(F.col(valid_col_name), 1).otherwise(0)).alias('valid'),
        F.sum(F.when(~F.col(valid_col_name), 1).otherwise(0)).alias('invalid'),
        F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias('null_count'),
        F.sum(F.when(F.col(column_name) == '', 1).otherwise(0)).alias('empty_count'),
        F.sum(F.when(F.col(column_name) == 'null', 1).otherwise(0)).alias('string_null_count')
    ).collect()[0]

    # Проблемные записи
    problematic_df = df_check.filter(~F.col(valid_col_name)).select(column_name)
    problematic_count = problematic_df.count()

    # Выводим информацию
    print(f"\nСТАТИСТИКА:")
    print(f"   Всего записей: {stats['total']:,}")
    print(f"   Валидных для преобразования в {type_name}: {stats['valid']:,}")
    print(f"   Невалидных: {stats['invalid']:,}")
    print(f"   Из них NULL: {stats['null_count']:,}")
    print(f"   Из них пустые строки: {stats['empty_count']:,}")
    print(f"   Из них строк 'null': {stats['string_null_count']:,}")

    # Для timestamp показываем примеры валидных и невалидных форматов
    if target_type == 'timestamp' and problematic_count > 0:
        print(f"\nОЖИДАЕМЫЙ ФОРМАТ TIMESTAMP: {timestamp_format}")
        print(f"   Пример: 2020-01-11 07:59:02")

    if problematic_count > 0:
        print(f"\nПРОБЛЕМНЫЕ ЗАПИСИ (первые 20):")
        problematic_df.show(10, truncate=False)

        # Показываем уникальные проблемные значения
        print(f"\nУНИКАЛЬНЫЕ ПРОБЛЕМНЫЕ ЗНАЧЕНИЯ:")
        problematic_df.distinct().show(10, truncate=False)

        # Для timestamp показываем статистику по длине строк
        if target_type == 'timestamp':
            print(f"\nАНАЛИЗ ПРОБЛЕМНЫХ TIMESTAMP:")
            problematic_with_length = problematic_df.withColumn('length', F.length(F.col(column_name)))
            problematic_with_length.groupBy('length').count().orderBy(F.desc('count')).show(10, truncate=False)
    else:
        print(f"\nНекорректных записей не найдено")

    # Сохраняем результаты в словарь
    result_stats = {
        'total': stats['total'],
        'valid': stats['valid'],
        'invalid': stats['invalid'],
        'null_count': stats['null_count'],
        'empty_count': stats['empty_count'],
        'string_null_count': stats['string_null_count'],
        'problematic_count': problematic_count,
        'valid_percent': (stats['valid'] / stats['total'] * 100) if stats['total'] > 0 else 0,
        'column_name': column_name,
        'target_type': target_type
    }

    return result_stats, problematic_df

# ФУНКЦИИ ИЗМЕНЕНИЯ ЗНАЧЕНИЯ

In [533]:
def fill_empty_values(df, column_name, replacement='NULL'):
    """
    Заменяет пустые строки в колонке

    Параметры:
    - df: Spark DataFrame
    - column_name: имя колонки
    - replacement: значение для замены ('NULL' для замены на NULL)

    Возвращает:
    - DataFrame с замененными значениями
    """
    if replacement == 'NULL':
        replacement = None

    df_result = df.withColumn(
        column_name,
        F.when(F.col(column_name) == '', F.lit(replacement))
         .otherwise(F.col(column_name))
    )

    replaced_count = df.filter(F.col(column_name) == '').count()
    print(f"Заменено пустых строк: {replaced_count}")
    print(f"Значение замены: {replacement if replacement is not None else 'NULL'}")

    return df_result

In [534]:
def fill_null_values(df, column_name, replacement, is_timestamp=False, timestamp_format='yyyy-MM-dd HH:mm:ss'):
    """
    Заменяет NULL значения в колонке

    Параметры:
    - df: Spark DataFrame
    - column_name: имя колонки
    - replacement: значение для замены
    - is_timestamp: True если колонка должна быть timestamp
    - timestamp_format: формат timestamp
    """
    if replacement == 'NULL':
        raise ValueError("Для замены NULL используйте fill_empty_values или fill_invalid_values с параметром 'NULL'")

    # Сначала преобразуем в нужный тип
    if is_timestamp:
        df = df.withColumn(column_name, F.to_timestamp(column_name, timestamp_format))

    # Заменяем NULL
    df_result = df.withColumn(
        column_name,
        F.when(F.col(column_name).isNull(), F.lit(replacement))
         .otherwise(F.col(column_name))
    )

    # Если это timestamp, преобразуем replacement
    if is_timestamp and isinstance(replacement, str):
        df_result = df_result.withColumn(
            column_name,
            F.to_timestamp(column_name, timestamp_format)
        )

    replaced_count = df.filter(F.col(column_name).isNull()).count()
    print(f"Заменено NULL значений: {replaced_count}")
    print(f"Значение замены: {replacement}")

    return df_result


# def fill_null_values(df, column_name, replacement):
#     """
#     Заменяет NULL значения в колонке

#     Параметры:
#     - df: Spark DataFrame
#     - column_name: имя колонки
#     - replacement: значение для замены (не может быть 'NULL')

#     Возвращает:
#     - DataFrame с замененными значениями
#     """
#     if replacement == 'NULL':
#         raise ValueError("Для замены NULL используйте fill_empty_values или fill_invalid_values с параметром 'NULL'")

#     df_result = df.withColumn(
#         column_name,
#         F.when(F.col(column_name).isNull(), F.lit(replacement))
#          .otherwise(F.col(column_name))
#     )

#     replaced_count = df.filter(F.col(column_name).isNull()).count()
#     print(f"Заменено NULL значений: {replaced_count}")
#     print(f"Значение замены: {replacement}")

#     return df_result

In [535]:
def fill_invalid_values(df, column_name, target_type, replacement,
                        timestamp_format='yyyy-MM-dd HH:mm:ss'):
    """
    Заменяет некорректные значения (которые не преобразуются в целевой тип)

    Параметры:
    - df: Spark DataFrame
    - column_name: имя колонки
    - target_type: целевой тип ('int', 'double', 'timestamp', etc.)
    - replacement: значение для замены ('NULL' для замены на NULL)
    - timestamp_format: формат timestamp (если target_type='timestamp')

    Возвращает:
    - DataFrame с замененными значениями
    """

    type_mapping = {
        'int': IntegerType(),
        'double': DoubleType(),
        'float': FloatType(),
        'long': LongType(),
        'string': StringType(),
        'timestamp': TimestampType()
    }

    if target_type not in type_mapping:
        raise ValueError(f"Неподдерживаемый тип: {target_type}")

    spark_type = type_mapping[target_type]

    # Сначала преобразуем, некорректные станут NULL
    if target_type == 'timestamp':
        df_converted = df.withColumn(
            column_name,
            F.to_timestamp(column_name, timestamp_format)
        )
    else:
        df_converted = df.withColumn(
            column_name,
            F.col(column_name).cast(spark_type)
        )

    # Считаем некорректные (стали NULL)
    invalid_count = df_converted.filter(
        F.col(column_name).isNull() &
        F.col(column_name).isNotNull()
    ).count()

    print(f"Найдено некорректных значений: {invalid_count}")

    # Заменяем NULL на replacement
    if replacement == 'NULL':
        # Оставляем как NULL
        print(f"Некорректные значения заменены на NULL")
        return df_converted
    else:
        # Заменяем на указанное значение
        if target_type == 'timestamp' and isinstance(replacement, str):
            replacement_value = F.to_timestamp(F.lit(replacement), timestamp_format)
        else:
            replacement_value = F.lit(replacement)

        df_result = df_converted.withColumn(
            column_name,
            F.when(F.col(column_name).isNull(), replacement_value)
             .otherwise(F.col(column_name))
        )

        print(f"Некорректные значения заменены на: {replacement}")
        return df_result

# Функция приведения всех значений колонки к единому формату

In [536]:
def uniform_column(df, column_name, target_type, timestamp_format='yyyy-MM-dd HH:mm:ss'):
    """
    Функция для приведения всей колонки к единому формату без замен значений

    Параметры:
    - df: Spark DataFrame
    - column_name: имя колонки
    - target_type: целевой тип ('int', 'double', 'float', 'long', 'string', 'timestamp')
    - timestamp_format: формат timestamp (для target_type='timestamp')

    Возвращает:
    - DataFrame с приведенной к единому формату колонкой
    """

    # Определяем тип для преобразования
    type_mapping = {
        'int': IntegerType(),
        'double': DoubleType(),
        'float': FloatType(),
        'long': LongType(),
        'string': StringType(),
        'timestamp': TimestampType()
    }

    if target_type not in type_mapping:
        raise ValueError(f"Неподдерживаемый тип: {target_type}. Используйте: {list(type_mapping.keys())}")

    spark_type = type_mapping[target_type]
    type_name = target_type.upper()

    print(f"\n{'='*70}")
    print(f"ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ")
    print(f"{'='*70}")
    print(f"Колонка: {column_name}")
    print(f"Целевой тип: {type_name}")
    print(f"{'='*70}")

    # Статистика до преобразования
    total_count = df.count()
    null_count_before = df.filter(F.col(column_name).isNull()).count()
    empty_count = df.filter(F.col(column_name) == '').count()

    print(f"\nСТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:")
    print(f"  Всего записей: {total_count:,}")
    print(f"  NULL значений: {null_count_before:,}")
    print(f"  Пустых строк: {empty_count:,}")

    # Преобразуем в целевой тип
    if target_type == 'timestamp':
        df_result = df.withColumn(
            column_name,
            F.to_timestamp(column_name, timestamp_format)
        )
    else:
        df_result = df.withColumn(
            column_name,
            F.col(column_name).cast(spark_type)
        )

    # Статистика после преобразования
    null_count_after = df_result.filter(F.col(column_name).isNull()).count()
    valid_count = total_count - null_count_after

    print(f"\nСТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:")
    print(f"  Успешно преобразовано: {valid_count:,}")
    print(f"  Стало NULL (некорректные значения): {null_count_after:,}")
    print(f"  Процент успеха: {valid_count/total_count*100:.2f}%")

    # Показываем примеры некорректных значений
    if null_count_after > null_count_before:
        new_null_count = null_count_after - null_count_before
        print(f"\nНекорректных значений (не удалось преобразовать): {new_null_count:,}")

    # Выводим тип
    print(f"\nИТОГОВЫЙ ТИП КОЛОНКИ:")
    df_result.printSchema()

    # Показываем примеры значений
    print(f"\nПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):")
    df_result.select(column_name).show(10, truncate=False)

    return df_result

# ЭТАП АНАЛИЗА И ОТЧИСТКИ ДАННЫХ

## Перегоняем данные в формат parquet

In [ ]:
(
    df_raw
        .write
        .mode("overwrite")
        .partitionBy("tx_time_days")
        .parquet("data/anti_fraud_all_data_df_filtered.parquet")
)

In [538]:
!hdfs dfs -ls -h data/anti_fraud_all_data_df_filtered.parquet

Found 31 items
-rw-r--r--   1 ubuntu hadoop          0 2026-06-15 15:52 data/anti_fraud_all_data_df_filtered.parquet/_SUCCESS
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 15:51 data/anti_fraud_all_data_df_filtered.parquet/tx_time_days=270
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 15:51 data/anti_fraud_all_data_df_filtered.parquet/tx_time_days=271
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 15:52 data/anti_fraud_all_data_df_filtered.parquet/tx_time_days=272
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 15:51 data/anti_fraud_all_data_df_filtered.parquet/tx_time_days=273
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 15:51 data/anti_fraud_all_data_df_filtered.parquet/tx_time_days=274
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 15:52 data/anti_fraud_all_data_df_filtered.parquet/tx_time_days=275
drwxr-xr-x   - ubuntu hadoop          0 2026-06-15 15:52 data/anti_fraud_all_data_df_filtered.parquet/tx_time_days=276
drwxr-xr-x   - ubuntu hadoop          0 2

In [539]:
df_filtered = spark.read.parquet("data/anti_fraud_all_data_df_filtered.parquet")

In [540]:
df_filtered.printSchema()

root
 |-- tranaction_id: string (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: string (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)



## Проверка даты транзакции

# Заполнение пустых ячеек минимальной датой

In [541]:
df_filtered = fix_24h_time(df_filtered, 'tx_datetime')

Заменено 91 записей с 24:00:00 на 00:00:00
Дата осталась без изменений


In [542]:
df_filtered = uniform_column(df_filtered, 'tx_datetime', 'timestamp')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: tx_datetime
Целевой тип: TIMESTAMP

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- tranaction_id: string (nullable = true)
 |-- tx_datetime: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: string (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+-------------------+
|tx_datetime        |
+-------------------+
|2020-06-06 16:49:01|
|2020-06-06 12:18:54|
|2020-06-06 15:46:50|
|2020-06-06 14:46:32|
|2020-06-06 13:53:32|
|2020-06-06 11:05:38|
|2020-06

In [543]:
# Использование
min_date, max_date, date_stats = analyze_dates(df_filtered, 'tx_datetime')


АНАЛИЗ ДАТ В КОЛОНКЕ: tx_datetime

ДИАПАЗОН ДАТ:
  Минимальная дата: 2020-05-18 00:00:00
  Максимальная дата: 2020-06-16 00:00:00
  Количество дней в диапазоне: 29

СТАТИСТИКА ПО ДАТАМ:
  Всего уникальных дат: 30
  Всего записей: 46,998,002

ПЕРВЫЕ 10 ДАТ (самые старые):
+----------+-------+----------+
|tx_date   |count  |null_count|
+----------+-------+----------+
|2020-05-18|1568075|0         |
|2020-05-19|1565651|0         |
|2020-05-20|1565383|0         |
|2020-05-21|1566271|0         |
|2020-05-22|1567515|0         |
|2020-05-23|1565373|0         |
|2020-05-24|1566447|0         |
|2020-05-25|1565408|0         |
|2020-05-26|1568796|0         |
|2020-05-27|1567687|0         |
+----------+-------+----------+
only showing top 10 rows


ПОСЛЕДНИЕ 10 ДАТ (самые новые):
+----------+-------+----------+
|tx_date   |count  |null_count|
+----------+-------+----------+
|2020-06-16|1564986|0         |
|2020-06-15|1567833|0         |
|2020-06-14|1567068|0         |
|2020-06-13|1567329|0       

In [544]:
df_filtered = fill_null_values(df_filtered, 'tx_datetime', replacement=min_date)

Заменено NULL значений: 0
Значение замены: 2020-05-18 00:00:00


In [545]:
analyze_column(df_filtered, 'tx_datetime', "timestamp")


АНАЛИЗ КОЛОНКИ: tx_datetime

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в TIMESTAMP: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'tx_datetime',
  'target_type': 'timestamp'},
 +-----------+
 |tx_datetime|
 +-----------+
 +-----------+)

In [546]:
df_filtered.printSchema()

root
 |-- tranaction_id: string (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: string (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)



# Проверяем остальные столбцы

# Преобразование tranaction_id в int

In [547]:
df_filtered = uniform_column(df_filtered, 'tranaction_id', 'int')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: tranaction_id
Целевой тип: INT

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- tranaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: string (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+-------------+
|tranaction_id|
+-------------+
|452699279    |
|452699280    |
|452699281    |
|452699282    |
|452699283    |
|452699284    |
|452699285    |
|452699286    |
|452699287    |
|452699288    |
+---

In [548]:
analyze_column(df_filtered, 'tranaction_id', 'int')


АНАЛИЗ КОЛОНКИ: tranaction_id

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в INT: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'tranaction_id',
  'target_type': 'int'},
 +-------------+
 |tranaction_id|
 +-------------+
 +-------------+)

In [549]:
# Переименовываем колонку
df_filtered = df_filtered.withColumnRenamed('tranaction_id', 'transaction_id')

# Проверяем
df_filtered.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: string (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)



# Преобразование customer_id в int

In [550]:
analyze_column(df_filtered, 'customer_id', 'int')


АНАЛИЗ КОЛОНКИ: customer_id

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в INT: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'customer_id',
  'target_type': 'int'},
 +-----------+
 |customer_id|
 +-----------+
 +-----------+)

In [551]:
df_filtered = uniform_column(df_filtered, 'customer_id', 'int')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: customer_id
Целевой тип: INT

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: string (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+-----------+
|customer_id|
+-----------+
|3          |
|6          |
|10         |
|10         |
|10         |
|10         |
|10         |
|11         |
|11         |
|12         |
+-----------+
only showing top

# Преобразование terminal_id в int

In [552]:
df_filtered = uniform_column(df_filtered, 'terminal_id', 'int')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: terminal_id
Целевой тип: INT

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: integer (nullable = true)
 |-- tx_amount: string (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+-----------+
|terminal_id|
+-----------+
|205        |
|809        |
|409        |
|380        |
|0          |
|663        |
|649        |
|337        |
|132        |
|13         |
+-----------+
only showing to

In [553]:
df_filtered = fill_invalid_values(df_filtered, 'terminal_id', 'int', replacement="-1")

Найдено некорректных значений: 0
Некорректные значения заменены на: -1


In [554]:
df_filtered = fill_empty_values(df_filtered, 'terminal_id', replacement="NULL")

Заменено пустых строк: 0
Значение замены: NULL


In [555]:
analyze_column(df_filtered, 'terminal_id', 'int')


АНАЛИЗ КОЛОНКИ: terminal_id

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в INT: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'terminal_id',
  'target_type': 'int'},
 +-----------+
 |terminal_id|
 +-----------+
 +-----------+)

# Преобразование tx_amount в double

In [556]:
df_filtered = uniform_column(df_filtered, 'tx_amount', 'double')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: tx_amount
Целевой тип: DOUBLE

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: string (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+---------+
|tx_amount|
+---------+
|13.28    |
|89.86    |
|83.28    |
|86.65    |
|155.96   |
|40.78    |
|103.58   |
|60.04    |
|14.37    |
|140.41   |
+---------+
only showing top 10 rows



In [557]:
analyze_column(df_filtered, 'tx_amount', 'double')


АНАЛИЗ КОЛОНКИ: tx_amount

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в DOUBLE: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'tx_amount',
  'target_type': 'double'},
 +---------+
 |tx_amount|
 +---------+
 +---------+)

# Преобразование tx_time_seconds в int

In [558]:
df_filtered = uniform_column(df_filtered, 'tx_time_seconds', 'int')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: tx_time_seconds
Целевой тип: INT

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: integer (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+---------------+
|tx_time_seconds|
+---------------+
|25030141       |
|25013934       |
|25026410       |
|25022792       |
|25019612       |
|25009538       |
|25011339       |
|25023747       |
|24999021

In [559]:
analyze_column(df_filtered, 'tx_time_seconds', 'int')


АНАЛИЗ КОЛОНКИ: tx_time_seconds

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в INT: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'tx_time_seconds',
  'target_type': 'int'},
 +---------------+
 |tx_time_seconds|
 +---------------+
 +---------------+)

# Проверяем колонку tx_time_days

In [560]:
df_filtered = uniform_column(df_filtered, 'tx_time_days', 'int')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: tx_time_days
Целевой тип: INT

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: integer (nullable = true)
 |-- tx_fraud: string (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+------------+
|tx_time_days|
+------------+
|289         |
|289         |
|289         |
|289         |
|289         |
|289         |
|289         |
|289         |
|289         |
|289         |
+------------+


In [561]:
analyze_column(df_filtered, 'tx_time_days', 'int')


АНАЛИЗ КОЛОНКИ: tx_time_days

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в INT: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'tx_time_days',
  'target_type': 'int'},
 +------------+
 |tx_time_days|
 +------------+
 +------------+)

# Проверяем колонку tx_fraud

In [562]:
df_filtered = uniform_column(df_filtered, 'tx_fraud', 'int')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: tx_fraud
Целевой тип: INT

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: integer (nullable = true)
 |-- tx_fraud: integer (nullable = true)
 |-- tx_fraud_scenario: string (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+--------+
|tx_fraud|
+--------+
|0       |
|1       |
|0       |
|0       |
|0       |
|0       |
|0       |
|0       |
|0       |
|0       |
+--------+
only showing top 10 rows



In [563]:
analyze_column(df_filtered, 'tx_fraud', 'int')


АНАЛИЗ КОЛОНКИ: tx_fraud

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в INT: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'tx_fraud',
  'target_type': 'int'},
 +--------+
 |tx_fraud|
 +--------+
 +--------+)

# Проверяем колонку tx_fraud_scenario

In [564]:
df_filtered = uniform_column(df_filtered, 'tx_fraud_scenario', 'int')


ПРИВЕДЕНИЕ КОЛОНКИ К ЕДИНОМУ ФОРМАТУ
Колонка: tx_fraud_scenario
Целевой тип: INT

СТАТИСТИКА ДО ПРЕОБРАЗОВАНИЯ:
  Всего записей: 46,998,002
  NULL значений: 0
  Пустых строк: 0

СТАТИСТИКА ПОСЛЕ ПРЕОБРАЗОВАНИЯ:
  Успешно преобразовано: 46,998,002
  Стало NULL (некорректные значения): 0
  Процент успеха: 100.00%

ИТОГОВЫЙ ТИП КОЛОНКИ:
root
 |-- transaction_id: integer (nullable = true)
 |-- tx_datetime: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- tx_amount: double (nullable = true)
 |-- tx_time_seconds: integer (nullable = true)
 |-- tx_fraud: integer (nullable = true)
 |-- tx_fraud_scenario: integer (nullable = true)
 |-- tx_time_days: integer (nullable = true)


ПРИМЕРЫ ЗНАЧЕНИЙ ПОСЛЕ ПРЕОБРАЗОВАНИЯ (первые 10):
+-----------------+
|tx_fraud_scenario|
+-----------------+
|0                |
|2                |
|0                |
|0                |
|0                |
|0                |
|0                |
|0 

In [565]:
analyze_column(df_filtered, 'tx_fraud_scenario', 'int')


АНАЛИЗ КОЛОНКИ: tx_fraud_scenario

СТАТИСТИКА:
   Всего записей: 46,998,002
   Валидных для преобразования в INT: 46,998,002
   Невалидных: 0
   Из них NULL: 0
   Из них пустые строки: 0
   Из них строк 'null': 0

Некорректных записей не найдено


({'total': 46998002,
  'valid': 46998002,
  'invalid': 0,
  'null_count': 0,
  'empty_count': 0,
  'string_null_count': 0,
  'problematic_count': 0,
  'valid_percent': 100.0,
  'column_name': 'tx_fraud_scenario',
  'target_type': 'int'},
 +-----------------+
 |tx_fraud_scenario|
 +-----------------+
 +-----------------+)

# Проверка условия: если tx_fraud=0, то tx_fraud_scenario не может принимать значение отличное от 0

In [566]:
# Проверяем условие: если tx_fraud = 0, то tx_fraud_scenario должен быть 0
df_check = df_filtered.withColumn(
    'fraud_condition_violation',
    F.when(
        (F.col('tx_fraud') == 0) & (F.col('tx_fraud_scenario') != 0),
        True
    ).otherwise(False)
)

# Показываем нарушения
violations = df_check.filter(F.col('fraud_condition_violation') == True)
violations_count = violations.count()

print(f"Найдено нарушений: {violations_count}")

if violations_count > 0:
    print("\nПримеры нарушений:")
    violations.select('tx_fraud', 'tx_fraud_scenario').show(20, truncate=False)
else:
    print("Условие выполнено: все записи с tx_fraud=0 имеют tx_fraud_scenario=0")

Найдено нарушений: 0
Условие выполнено: все записи с tx_fraud=0 имеют tx_fraud_scenario=0


In [567]:
df_filtered.show()

+--------------+-------------------+-----------+-----------+---------+---------------+--------+-----------------+------------+
|transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_fraud|tx_fraud_scenario|tx_time_days|
+--------------+-------------------+-----------+-----------+---------+---------------+--------+-----------------+------------+
|     452699279|2020-06-06 16:49:01|          3|        205|    13.28|       25030141|       0|                0|         289|
|     452699280|2020-06-06 12:18:54|          6|        809|    89.86|       25013934|       1|                2|         289|
|     452699281|2020-06-06 15:46:50|         10|        409|    83.28|       25026410|       0|                0|         289|
|     452699282|2020-06-06 14:46:32|         10|        380|    86.65|       25022792|       0|                0|         289|
|     452699283|2020-06-06 13:53:32|         10|          0|   155.96|       25019612|       0|                

In [568]:
df_filtered.count()

46998002

# Сохраняем результат работы в файл формата паркет

In [569]:
# Проверяем количество партиций перед сохранением
current_partitions = df_filtered.rdd.getNumPartitions()
print(f"Текущее количество партиций: {current_partitions}")

Текущее количество партиций: 10


In [570]:
# Узнать количество уникальных значений tx_time_days
unique_days = df_filtered.select('tx_time_days').distinct().count()
print(f"Уникальных значений tx_time_days: {unique_days}")  # Будет 31

Уникальных значений tx_time_days: 30


In [571]:
# Ваше имя бакета
bucket_name = "otus-bucket-b1g4ki09n8igs1si54v2"

# Сохраняем DataFrame в Parquet
# Сохраняем данные по партициям
df_filtered.write \
    .mode("overwrite") \
    .parquet(f"s3a://{bucket_name}/{file_data}.parquet")

print("Данные успешно сохранены в бакет!")

Данные успешно сохранены в бакет!


In [10]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
import subprocess

def process_all_files():

    # Получаем список всех 40 файлов из HDFS
    result = subprocess.run(
        ["hdfs", "dfs", "-ls", "/user/ubuntu/data/"],
        capture_output=True, text=True
    )

    files = []
    for line in result.stdout.strip().split('\n'):
        if '.txt' in line:
            file_path = line.split()[-1]
            file_name = file_path.split('/')[-1].replace('.txt', '')
            files.append(file_name)

    files = sorted(files)
    print(f"\n{'='*70}")
    print(f"НАЙДЕНО ФАЙЛОВ ДЛЯ ОБРАБОТКИ: {len(files)}")
    print(f"{'='*70}")

    results = {}

    for file_date in files:
        print(f"\n{'='*70}")
        print(f"ОБРАБОТКА ФАЙЛА: {file_date}")
        print(f"{'='*70}")

        try:
            # 1. ЧИТАЕМ ИСХОДНЫЙ ФАЙЛ
            print(f"\n1. ЧТЕНИЕ ИСХОДНОГО ФАЙЛА: data/{file_date}.txt")
            rdd = spark.sparkContext.textFile(f"data/{file_date}.txt")
            first_line = rdd.first()

            columns = ['tranaction_id', 'tx_datetime', 'customer_id', 'terminal_id',
                       'tx_amount', 'tx_time_seconds', 'tx_time_days', 'tx_fraud', 'tx_fraud_scenario']

            if first_line.startswith("#"):
                data = rdd.filter(lambda x: not x.startswith("#")).map(lambda x: x.split(","))
                print(f"  Формат: с заголовком #")
            else:
                data = rdd.map(lambda x: x.split(","))
                print(f"  Формат: без заголовка")

            df_raw = spark.createDataFrame(data, schema=columns)
            raw_count = df_raw.count()
            print(f"  Прочитано записей: {raw_count:,}")

            # 2. ПАРТИЦИОНИРОВАНИЕ (КАК В ТВОЕМ СКРИПТЕ)
            print(f"\n2. СОХРАНЕНИЕ С ПАРТИЦИОНИРОВАНИЕМ ПО tx_time_days")
            (
                df_raw
                    .write
                    .mode("overwrite")
                    .partitionBy("tx_time_days")
                    .parquet(f"data/{file_date}_partitioned.parquet")
            )
            print(f"  Сохранено с партиционированием в: data/{file_date}_partitioned.parquet")

            # 3. ЧИТАЕМ ПАРТИЦИОНИРОВАННЫЕ ДАННЫЕ (для быстрой обработки)
            print(f"\n3. ЧТЕНИЕ ПАРТИЦИОНИРОВАННЫХ ДАННЫХ")
            df = spark.read.parquet(f"data/{file_date}_partitioned.parquet")
            print(f"  Прочитано партиций: {df.rdd.getNumPartitions()}")

            # 4. ОБРАБОТКА ДАННЫХ
            print(f"\n4. ОБРАБОТКА ДАННЫХ")

            # Исправляем время 24:00:00
            df = df.withColumn('tx_datetime', F.regexp_replace('tx_datetime', '24:00:00', '00:00:00'))

            # Преобразуем дату в timestamp
            df = df.withColumn('tx_datetime', F.to_timestamp('tx_datetime', 'yyyy-MM-dd HH:mm:ss'))

            # Пустые строки в terminal_id -> NULL
            df = df.withColumn('terminal_id',
                               F.when(F.col('terminal_id') == '', F.lit(None))
                                .otherwise(F.col('terminal_id')))

            # Преобразуем числовые колонки в int
            int_columns = ['tranaction_id', 'customer_id', 'terminal_id',
                           'tx_time_seconds', 'tx_time_days', 'tx_fraud', 'tx_fraud_scenario']
            for col in int_columns:
                df = df.withColumn(col, F.col(col).cast(IntegerType()))

            # Преобразуем сумму в double
            df = df.withColumn('tx_amount', F.col('tx_amount').cast(DoubleType()))

            # Переименовываем колонку
            df = df.withColumnRenamed('tranaction_id', 'transaction_id')

            # Проверка условия fraud
            violations = df.filter((F.col('tx_fraud') == 0) & (F.col('tx_fraud_scenario') != 0)).count()
            if violations > 0:
                print(f"  ВНИМАНИЕ: Найдено {violations} нарушений (fraud=0 но scenario!=0)")
            else:
                print(f"  Проверка fraud: OK")

            final_count = df.count()
            print(f"  После обработки: {final_count:,} записей")

            # 5. ФИНАЛЬНОЕ СОХРАНЕНИЕ В S3
            print(f"\n5. СОХРАНЕНИЕ В S3")
            bucket_name = "otus-bucket-b1g4ki09n8igs1si54v2"
            output_path = f"s3a://{bucket_name}/{file_date}.parquet"

            (
                df.write
                    .mode("overwrite")
                    .parquet(output_path)
            )

            print(f"  Сохранено в: {output_path}")
            print(f"  Записей: {final_count:,}")

            results[file_date] = {
                'raw_count': raw_count,
                'final_count': final_count,
                'violations': violations
            }

        except Exception as e:
            print(f"\n  ОШИБКА: {e}")
            results[file_date] = {'error': str(e)}

    # ФИНАЛЬНЫЙ ОТЧЕТ
    print(f"\n{'='*70}")
    print("ИТОГОВЫЙ ОТЧЕТ ПО ОБРАБОТКЕ 40 ФАЙЛОВ")
    print(f"{'='*70}")

    total_raw = 0
    total_final = 0
    total_violations = 0

    for f, data in results.items():
        if 'error' in data:
            print(f"  ОШИБКА {f}: {data['error']}")
        else:
            raw = data.get('raw_count', 0)
            final = data.get('final_count', 0)
            viol = data.get('violations', 0)
            total_raw += raw
            total_final += final
            total_violations += viol
            print(f"  OK {f}: сырых={raw:,} -> после обработки={final:,} | нарушений={viol}")

    print(f"\n{'='*70}")
    print(f"ВСЕГО:")
    print(f"  Сырых записей: {total_raw:,}")
    print(f"  После обработки: {total_final:,}")
    print(f"  Нарушений fraud логики: {total_violations}")
    print(f"{'='*70}")

    return results

# ЗАПУСК
all_results = process_all_files()


НАЙДЕНО ФАЙЛОВ ДЛЯ ОБРАБОТКИ: 30

ОБРАБОТКА ФАЙЛА: 2020-06-17

1. ЧТЕНИЕ ИСХОДНОГО ФАЙЛА: data/2020-06-17.txt
  Формат: с заголовком #
  Прочитано записей: 46,983,063

2. СОХРАНЕНИЕ С ПАРТИЦИОНИРОВАНИЕМ ПО tx_time_days
  Сохранено с партиционированием в: data/2020-06-17_partitioned.parquet

3. ЧТЕНИЕ ПАРТИЦИОНИРОВАННЫХ ДАННЫХ
  Прочитано партиций: 10

4. ОБРАБОТКА ДАННЫХ
  Проверка fraud: OK
  После обработки: 46,983,063 записей

5. СОХРАНЕНИЕ В S3
  Сохранено в: s3a://otus-bucket-b1g4ki09n8igs1si54v2/2020-06-17.parquet
  Записей: 46,983,063

ОБРАБОТКА ФАЙЛА: 2020-07-17

1. ЧТЕНИЕ ИСХОДНОГО ФАЙЛА: data/2020-07-17.txt
  Формат: с заголовком #
  Прочитано записей: 47,000,292

2. СОХРАНЕНИЕ С ПАРТИЦИОНИРОВАНИЕМ ПО tx_time_days
  Сохранено с партиционированием в: data/2020-07-17_partitioned.parquet

3. ЧТЕНИЕ ПАРТИЦИОНИРОВАННЫХ ДАННЫХ
  Прочитано партиций: 10

4. ОБРАБОТКА ДАННЫХ
  Проверка fraud: OK
  После обработки: 47,000,292 записей

5. СОХРАНЕНИЕ В S3
  Сохранено в: s3a://otus-bucke

In [ ]:
import subprocess
from pyspark.sql import functions as F

# Получаем список файлов из бакета
result = subprocess.run(
    ["hadoop", "fs", "-ls", "s3a://otus-bucket-b1g4ki09n8igs1si54v2/"],
    capture_output=True, text=True
)

files = []
for line in result.stdout.strip().split('\n'):
    if '.parquet' in line and 'SUCCESS' not in line:
        file_path = line.split()[-1]
        files.append(file_path)

print(f"Найдено parquet файлов: {len(files)}")

# Читаем каждый файл и считаем
total = 0
file_counts = {}
total_tx_datetime = 0
total_tx_transaction_id = 0
total_tx_customer_id  = 0
total_tx_terminal_id = 0
total_tx_time_seconds = 0
total_tx_amount = 0
total_tx_fraud = 0
total_tx_time_days = 0
total_tx_fraud_scenario = 0

for f in files:
    try:
        df = spark.read.parquet(f)
        null_values_tx_datetime = df.filter(df['tx_datetime'].isNull()).count()
        null_values_tx_transaction_id = df.filter(df['transaction_id'].isNull()).count()
        null_values_tx_customer_id = df.filter(df['customer_id'].isNull()).count()
        null_values_tx_terminal_id = df.filter(df['terminal_id'].isNull()).count()
        null_values_tx_amount = df.filter(df['tx_amount'].isNull()).count()
        null_values_tx_time_seconds = df.filter(df['tx_time_seconds'].isNull()).count()
        null_values_tx_fraud = df.filter(df['tx_fraud'].isNull()).count()
        null_values_tx_fraud_scenario = df.filter(df['tx_fraud_scenario'].isNull()).count()
        null_values_tx_tx_time_days = df.filter(df['tx_time_days'].isNull()).count()

        count = df.count()
        total += count
        total_tx_datetime += null_values_tx_datetime
        total_tx_transaction_id += null_values_tx_transaction_id
        total_tx_customer_id  += null_values_tx_customer_id
        total_tx_terminal_id += null_values_tx_terminal_id
        total_tx_time_seconds += null_values_tx_time_seconds
        total_tx_amount += null_values_tx_amount
        total_tx_fraud += null_values_tx_fraud
        total_tx_time_days += null_values_tx_tx_time_days
        total_tx_fraud_scenario += null_values_tx_fraud_scenario


        file_name = f.split('/')[-1]
        file_counts[file_name] = count
        print(f"{file_name}: {count:,} записей")
    except Exception as e:
        print(f"Ошибка при чтении {f}: {e}")

print(f"\nИТОГО: {total:,} записей")

Найдено parquet файлов: 40
2019-08-22.parquet: 46,988,418 записей
2019-09-21.parquet: 46,994,586 записей
2019-10-21.parquet: 46,994,432 записей
2019-11-20.parquet: 46,992,239 записей
2019-12-20.parquet: 46,994,937 записей
2020-01-19.parquet: 46,986,197 записей
2020-02-18.parquet: 46,994,271 записей
2020-03-19.parquet: 46,990,424 записей
2020-04-18.parquet: 47,001,235 записей
2020-05-18.parquet: 46,998,002 записей
2020-06-17.parquet: 46,983,063 записей
2020-07-17.parquet: 47,000,292 записей
2020-08-16.parquet: 47,003,159 записей
2020-09-15.parquet: 46,999,865 записей
2020-10-15.parquet: 47,001,238 записей
2020-11-14.parquet: 46,995,020 записей
2020-12-14.parquet: 46,983,905 записей
2021-01-13.parquet: 46,993,755 записей


transaction_id|        tx_datetime|customer_id|terminal_id|tx_amount|tx_time_seconds|tx_fraud|tx_fraud_scenario|tx_time_days

In [24]:
df_filtered.filter(df_filtered['tx_datetime'].isNull()).count()

0